# Module 09B - Pretraining

Use this notebook after Module 09's `TransformerLM` is working. The goal is to make the language-model training data concrete: one token stream becomes train/validation streams, `(B, T)` shifted batches, `(B, T, V)` logits, and one scalar cross-entropy loss.

1. Read the lesson page (`docs/modules/09b-pretraining.md`).
2. Open this notebook with `./notebook.sh 09b`.
3. Answer the `Question:` / `Answer:` cells below.
4. When you're ready, ask a coding agent to grade your notebook.

Partial work is fine. Blank `Answer: ""` strings are skipped, not counted wrong. If you'd like a hint instead of a grade, write the request inline in the answer string and the agent will tutor first.

In [ ]:
from __future__ import annotations

import math
import subprocess
import sys
from pathlib import Path

import torch

import g2c
from g2c.pretraining import get_lm_batch, lm_cross_entropy, split_token_stream
from g2c.transformer import TransformerLM

_ = torch.manual_seed(0)
repo_root = Path(g2c.__file__).resolve().parents[1]
print("repo root:", repo_root)


## Before the Notebook

`split_token_stream` and `get_lm_batch` are implemented for you. Implement `lm_cross_entropy` before working through the later cells.

In [ ]:
"Run from the terminal: .venv/bin/python -m pytest tests/test_pretraining_setup.py -x"
"Question: Which setup test is the next one failing, and what shape contract does it point at?"
"Answer: "


In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_pretraining_setup.py"],
    cwd=repo_root,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, "Module 09B pretraining setup tests are not passing yet."
print("Module 09B pretraining setup tests passed.")


## Exercise 1 - Shift a Toy Stream

Start with a sequence where the answer is visible by inspection.

In [ ]:
ids = torch.tensor([10, 11, 12, 13, 14, 15])
start = 1
T = 3
x = ids[start : start + T]
y = ids[start + 1 : start + T + 1]
print("ids:", ids.tolist())
print("x:", x.tolist())
print("y:", y.tolist())


In [ ]:
"Question: Why is y[t] the training target for x[t]?"
"Answer: "


## Exercise 2 - Split a Token Stream

A language-model split preserves order inside each split. Randomness comes from sampling windows later, not from shuffling individual tokens.

In [ ]:
toy_stream = torch.arange(30)
train_ids, val_ids = split_token_stream(toy_stream, train_fraction=0.8)
print("train:", train_ids.tolist())
print("val:", val_ids.tolist())
print("train tokens:", len(train_ids))
print("val tokens:", len(val_ids))


In [ ]:
"Question: Why would shuffling individual token IDs break next-token prediction?"
"Answer: "


## Exercise 3 - Sample Multi-Position Batches

`get_lm_batch` samples random contiguous windows. With `torch.arange`, every target should equal the matching input plus one.

In [ ]:
ids = torch.arange(30)
xb, yb = get_lm_batch(
    ids,
    batch_size=4,
    context_length=6,
    generator=torch.Generator().manual_seed(0),
)
print("x shape:", tuple(xb.shape))
print("y shape:", tuple(yb.shape))
print("x:\n", xb)
print("y:\n", yb)
print("y == x + 1:", torch.equal(yb, xb + 1))
assert xb.shape == (4, 6)
assert yb.shape == (4, 6)
assert torch.equal(yb, xb + 1)


In [ ]:
"Question: Why does this one (B, T) batch contain B*T classification examples?"
"Answer: "


## Exercise 4 - Language-Model Cross-Entropy

The loss is ordinary cross-entropy after flattening positions.

In [ ]:
B, T, V = 2, 4, 7
logits = torch.zeros(B, T, V)
targets = torch.randint(0, V, (B, T), generator=torch.Generator().manual_seed(0))
loss = lm_cross_entropy(logits, targets)
print("logits shape:", tuple(logits.shape))
print("targets shape:", tuple(targets.shape))
print("flat logits shape:", (B * T, V))
print("flat targets shape:", (B * T,))
print("loss:", float(loss))
print("log(V):", math.log(V))
assert abs(loss.item() - math.log(V)) < 1e-5


In [ ]:
"Question: What reshape turns logits from (B, T, V) into the shape CrossEntropyLoss expects?"
"Answer: "


## Exercise 5 - Baselines for Different Vocabulary Sizes

The uniform-loss baseline rises with vocabulary size because uniform guessing has more classes to choose from.

In [ ]:
for V in [256, 512, 1024]:
    loss = math.log(V)
    print(f"V={V:4d}  log(V)={loss:.3f}  perplexity={math.exp(loss):.0f}")


In [ ]:
"Question: Why can a larger vocabulary start with larger loss but still be useful?"
"Answer: "


## Exercise 6 - Random Transformer Sanity Check

A random model should usually start near `log(V)`. This does not mean it is good; it means the objective is wired plausibly before training.

In [ ]:
torch.manual_seed(0)
vocab_size = 64
model = TransformerLM(
    vocab_size=vocab_size,
    embedding_dim=16,
    num_layers=1,
    num_heads=2,
    max_seq_len=8,
)
ids = torch.randint(0, vocab_size, (500,), generator=torch.Generator().manual_seed(1))
xb, yb = get_lm_batch(ids, batch_size=8, context_length=8, generator=torch.Generator().manual_seed(2))
logits = model(xb)
loss = lm_cross_entropy(logits, yb)
print("logits shape:", tuple(logits.shape))
print("loss:", float(loss))
print("log(V):", math.log(vocab_size))


In [ ]:
"Question: If the random-model loss is far below log(V), what are two possible explanations?"
"Answer: "


When complete, ask a coding agent to grade your Module 09B notebook. Partial work is fine: the agent should grade answered questions and implemented sections, then skip blank prompts.